<a href="https://colab.research.google.com/github/Ritesh-sudo/ufld-edge-lka/blob/main/clrnet_nms_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLRNet lane-NMS check: pure-PyTorch port vs the real CUDA op

Runs on a **GPU Colab runtime**. It builds upstream CLRNet's CUDA NMS extension (only the op, not the old mmcv stack), feeds it
the *real* NMS inputs saved from CLRNet on 400 TuSimple test images (`tests/data/clrnet_nms_cases.npz`), and checks that it keeps
exactly the same lanes as `third_party/clrnet/nms.py`. Also runs random stress cases.

The upstream .cu/.cpp use PyTorch APIs removed in current releases (`Tensor::data<T>()`, `.type().is_cuda()`); the build cell
patches **only those API names**. The kernel logic is untouched, and the cell prints the diff so you can confirm.

In [1]:
GITHUB_REPO = "Ritesh-sudo/ufld-edge-lka"   # same as notebooks/colab_train.ipynb; add a GITHUB_TOKEN Colab secret if the repo is private
import subprocess
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token + '@' if token else ''}github.com/{GITHUB_REPO}.git"
r = subprocess.run(["git", "clone", "-q", url, "/content/repo"], capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("clone failed:\n" + (r.stderr.replace(token, "***") if token else r.stderr))
subprocess.run(["git", "clone", "-q", "https://github.com/Turoad/CLRNet", "/content/CLRNet_up"], check=True)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU: Runtime > Change runtime type > T4 GPU")

GPU 0: Tesla T4 (UUID: GPU-0e44fe6b-32ec-7a6a-f6ed-33e7d0ef69ac)



In [2]:
import re, pathlib, subprocess
src = pathlib.Path('/content/CLRNet_up/clrnet/ops/csrc'); dst = pathlib.Path('/content/nms_build'); dst.mkdir(exist_ok=True)
for f in ('nms.cpp', 'nms_kernel.cu'):
    t = (src / f).read_text()
    t = t.replace('boxes.data<scalar_t>()', 'boxes.data_ptr<scalar_t>()').replace('idx.data<int64_t>()', 'idx.data_ptr<int64_t>()')
    t = t.replace('mask.data<int64_t>()', 'mask.data_ptr<int64_t>()').replace('keep.data<int64_t>()', 'keep.data_ptr<int64_t>()')
    t = t.replace('parent_object_index.data<int64_t>()', 'parent_object_index.data_ptr<int64_t>()').replace('num_to_keep.data<int64_t>()', 'num_to_keep.data_ptr<int64_t>()')
    t = t.replace('x.type().is_cuda()', 'x.is_cuda()').replace('AT_DISPATCH_FLOATING_TYPES(boxes.type(),', 'AT_DISPATCH_FLOATING_TYPES(boxes.scalar_type(),')
    (dst / f).write_text(t)
print(subprocess.run(['diff', '-u', str(src/'nms.cpp'), str(dst/'nms.cpp')], capture_output=True, text=True).stdout)
print(subprocess.run(['diff', '-u', str(src/'nms_kernel.cu'), str(dst/'nms_kernel.cu')], capture_output=True, text=True).stdout)

--- /content/CLRNet_up/clrnet/ops/csrc/nms.cpp	2026-11-12 21:10:32.666048440 +0000
+++ /content/nms_build/nms.cpp	2026-11-12 21:10:32.705052258 +0000
@@ -37,7 +37,7 @@
         float nms_overlap_thresh,
         unsigned long top_k);
 
-#define CHECK_CUDA(x) AT_ASSERTM(x.type().is_cuda(), #x " must be a CUDA tensor")
+#define CHECK_CUDA(x) AT_ASSERTM(x.is_cuda(), #x " must be a CUDA tensor")
 #define CHECK_CONTIGUOUS(x) AT_ASSERTM(x.is_contiguous(), #x " must be contiguous")
 #define CHECK_INPUT(x) CHECK_CUDA(x); CHECK_CONTIGUOUS(x)
 

--- /content/CLRNet_up/clrnet/ops/csrc/nms_kernel.cu	2026-11-12 21:10:32.666048440 +0000
+++ /content/nms_build/nms_kernel.cu	2026-11-12 21:10:32.705052258 +0000
@@ -168,12 +168,12 @@
   CHECK_CONTIGUOUS(idx);
   CHECK_CONTIGUOUS(mask);
 
-  AT_DISPATCH_FLOATING_TYPES(boxes.type(), "nms_cuda_forward", ([&] {
+  AT_DISPATCH_FLOATING_TYPES(boxes.scalar_type(), "nms_cuda_forward", ([&] {
     nms_kernel<<<blocks, threads>>>(boxes_num,
                      

In [7]:
!pip install -q ninja

In [9]:
import torch
from torch.utils.cpp_extension import load
nms_impl = load(name='nms_impl', sources=[str(dst/'nms.cpp'), str(dst/'nms_kernel.cu')], verbose=False)
print('built; torch', torch.__version__, 'cuda', torch.version.cuda)

built; torch 2.14.1+cu130 cuda 13.0


In [10]:
import sys, numpy as np
sys.path.insert(0, '/content/repo')
from third_party.clrnet import lane_nms

def cuda_nms(boxes, scores, thr, top_k):
    keep, num, _ = nms_impl.nms_forward(boxes.cuda().contiguous(), scores.cuda(), float(thr), int(top_k))
    return keep[:num].cpu().tolist()

z = np.load('/content/repo/tests/data/clrnet_nms_cases.npz')
bo, thr, top_k = z['box_off'], float(z['nms_thres']), int(z['top_k'])
n_cmp = n_bad = 0
for i in range(len(bo) - 1):
    b, s = torch.from_numpy(z['boxes'][bo[i]:bo[i+1]]), torch.from_numpy(z['scores'][bo[i]:bo[i+1]])
    if len(b) == 0: continue
    keep, num = lane_nms(b, s, thr, top_k)
    a, c = keep[:num].tolist(), cuda_nms(b, s, thr, top_k)
    n_cmp += 1; n_bad += (a != c)
    if a != c and n_bad <= 5: print('MISMATCH', z['names'][i], 'torch', a, 'cuda', c)
print(f'real inputs: {n_cmp} images compared, {n_bad} mismatches')

real inputs: 400 images compared, 0 mismatches


In [11]:
rng = np.random.default_rng(0)
def rand_lanes(n):
    b = np.zeros((n, 77), np.float32)
    for i in range(n):
        if i and rng.random() < 0.4:
            b[i] = b[rng.integers(0, i)]; b[i, 5:] += rng.normal(0, 8, 72); continue
        st = rng.integers(0, 40); b[i, 2] = st / 71; b[i, 4] = rng.integers(5, 72 - st)
        b[i, 5:] = rng.uniform(0, 799) + rng.uniform(-6, 6) * np.arange(72) * rng.uniform(0, 1)
    return b
bad = 0
for t in range(300):
    b = torch.from_numpy(rand_lanes(int(rng.integers(2, 120)))); s = torch.rand(len(b))
    for th in (10., 50., 150.):
        keep, num = lane_nms(b, s, th, 5)
        bad += keep[:num].tolist() != cuda_nms(b, s, th, 5)
print('random stress: mismatches =', bad, 'of', 300 * 3)
assert n_bad == 0 and bad == 0, 'NMS port does NOT match CUDA - do not use for benchmarks'
print('PASS: pure-PyTorch NMS keeps exactly the same lanes as the CUDA op')

random stress: mismatches = 0 of 900
PASS: pure-PyTorch NMS keeps exactly the same lanes as the CUDA op


Paste the printed counts into the README's CLRNet section and `results/clrnet_nms_check.txt`.